# Kaggle Pipeline — runner

Thin notebook to run the [Kaggle-Pipeline](https://github.com/HkHk2Prod/Kaggle-Pipeline) package on Kaggle.

**Workflow:** run the setup cell, edit the few metaparameters in the config cell, optionally explore the data, then train.

**Internet:** the setup cell *auto-detects* connectivity. With Internet on (Settings → Internet → On) it `pip install`s the package from GitHub; with Internet off it imports the repo from a Kaggle Dataset you've attached (see *Offline use* at the bottom). The same notebook therefore runs unchanged in both Playground-series and code (internet-off) competitions.

In [ ]:
# 1. Make `kaggle_pipeline` importable — automatically, with or without internet.
#
#   Internet ON  -> pip install from GitHub (pin to a tag, e.g. @v0.1.0, for
#                   reproducibility).
#   Internet OFF -> import the repo from a Kaggle Dataset you've attached
#                   (see "Offline use" at the bottom). No edits needed either way.
import importlib.util
import socket
import subprocess
import sys
from pathlib import Path

REPO_URL = "git+https://github.com/HkHk2Prod/Kaggle-Pipeline.git"


def _has_internet(host="github.com", port=443, timeout=5.0):
    """True if a TCP connection to host:port succeeds within `timeout` seconds."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def _add_offline_package():
    """Put a repo attached as a Kaggle Dataset on sys.path. Returns True on success."""
    root = Path("/kaggle/input")
    if not root.exists():
        return False
    # Find the kaggle_pipeline package itself a few levels deep, so any dataset
    # name / ZIP nesting works (e.g. <ds>/src/kaggle_pipeline or
    # <ds>/Kaggle-Pipeline-main/src/kaggle_pipeline), and add its parent dir.
    for pattern in ("*/kaggle_pipeline", "*/*/kaggle_pipeline", "*/*/*/kaggle_pipeline"):
        for pkg in sorted(root.glob(pattern)):
            if (pkg / "__init__.py").is_file():
                parent = str(pkg.parent)
                if parent not in sys.path:
                    sys.path.insert(0, parent)
                print(f"[setup] offline: imported kaggle_pipeline from {parent}")
                return True
    return False


if importlib.util.find_spec("kaggle_pipeline") is not None:
    print("[setup] kaggle_pipeline already available")
elif _has_internet():
    print("[setup] internet detected -> pip install from GitHub")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", REPO_URL], check=True)
elif not _add_offline_package():
    raise RuntimeError(
        "No internet, and no offline copy of the package was found. Publish this "
        "repo as a Kaggle Dataset (run scripts/publish_dataset.py, or GitHub: "
        "Code -> Download ZIP then create a Dataset) and attach it via Add Input, "
        "then re-run this cell. See 'Offline use' at the bottom of the notebook."
    )

In [ ]:
# 2. The few metaparameters you tune per competition.
#
# Nothing here is strictly required on Kaggle. The data directory is found by
# scanning the attached inputs for the one containing train/test CSVs, and the
# problem-definition fields (target / task / scoring / ...) and CSV filenames are
# autodetected from the data once it loads. Every autodetected value is printed
# as an `[autodetect] ...` line so the run stays reproducible from its log.
#
# Available cfg options
# ---------------------
#   competition       Kaggle competition slug. Optional: only needed to pick the
#                     right input when several attached inputs contain CSVs (or to
#                     derive the Colab data path). Otherwise data_dir is autodetected.
#   target            Target column name(s). None -> the last non-id column of train.
#   id_col            Id column name(s): kept out of the model, reused in the
#                     submission. Default "id".
#   task              "classification" or "regression". None -> inferred from the
#                     target dtype (text / few-valued integers -> classification,
#                     otherwise regression).
#   scoring           CV metric: "roc_auc", "balanced_accuracy" or
#                     "neg_root_mean_squared_error". None -> "roc_auc" for binary
#                     targets, "balanced_accuracy" for multiclass.
#   prediction_aim    Submission format: "category" (hard labels) or "probability".
#                     None -> "probability" when the task is classification.
#   feature_expressions  pandas.eval strings that add new columns; bools -> 0/1.
#   n_steps           Number of model-search batches to run.
#   num_models        Global leaderboard capacity.
#   step_batch_size   Models evaluated per step (run in parallel).
#   n_workers         joblib n_jobs (-1 = all cores).
#   ensemble_length   Max base models stacked into the final ensemble.
#   ensemble_min_repr Minimum models kept per class in the ensemble.
#   cv_splits, cv_seed   Cross-validation folds and the seed for their splits.
#   speed_up          True -> subsample the data for a fast debug run.
#   max_running_time  Stop the search after this many seconds (Kaggle caps at 12h).
#   seed              Global RNG seed; None = non-reproducible.
#   train_csv / test_csv / sample_csv  CSV filenames inside the data dir. None ->
#                     found by searching for names containing train / test / sample.
#   data_dir / storage_dir   Path overrides. data_dir is required when running
#                     locally; on Kaggle it is autodetected from the attached inputs.
#
# On Kaggle a minimal config can be as short as `Config()`. Below the fields are
# set explicitly for this competition; comment any of them out to autodetect it.
from kaggle_pipeline import Config

cfg = Config(
    competition="playground-series-s6e4",  # optional; disambiguates multiple inputs
    target="Irrigation_Need",      # or omit -> last column of train.csv
    id_col="id",
    task="classification",         # or omit -> inferred from the target dtype
    scoring="balanced_accuracy",   # or omit -> roc_auc (binary) / balanced_accuracy
    prediction_aim="category",     # or omit -> "probability" for classification
    feature_expressions=[
        "soil_lt_25  = Soil_Moisture < 25",
        "temp_gt_30  = Temperature_C > 30",
        "rain_lt_300 = Rainfall_mm < 300",
        "wind_gt_10  = Wind_Speed_kmh > 10",
    ],
    n_steps=10,
    num_models=300,
    speed_up=False,   # set True for a fast debug run on a data subset
    max_running_time=43200,
)

## Explore (optional)

Exploratory data analysis is fully decoupled from training — run this cell only when you want to look at the data. It renders metadata, correlation heatmaps and pairwise plots, and does not affect the training run below. Skip it entirely when training.

In [ ]:
from kaggle_pipeline import analyze

analyze(cfg)

## Train

Loads data, searches models, ensembles, and writes `submission.csv`.

In [ ]:
from kaggle_pipeline import run

submission_path = run(cfg)
print("Wrote", submission_path)

---
## Offline use (no internet)

The setup cell at the top auto-detects connectivity, so **no code change is needed** for internet-off (code) competitions — when there is no connection it imports the package from a Kaggle Dataset instead of pip-installing it. Publish this repository as a Dataset once:

- **Automated:** from a clone of the repo run `python scripts/publish_dataset.py` (needs `pip install kaggle` and a Kaggle API token at `~/.kaggle/kaggle.json`). It creates the dataset the first time and uploads a new version on later runs.
- **Manual:** on GitHub use *Code → Download ZIP*, then create a Kaggle **Dataset** from the ZIP.

Either way, attach the dataset to this notebook via *Add Input*. The setup cell searches `/kaggle/input` (a few levels deep) for the `kaggle_pipeline` package, so the dataset's name and internal nesting don't matter.